# Tutorial 03: dtypes & Devices (CPU / GPU)

`float32` vs `float64`, `int64`, moving tensors with `.to(device)`.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {device}')

## Data types matter for memory & precision

In [ ]:
dtypes = [torch.float16, torch.float32, torch.float64, torch.int32, torch.int64, torch.bool]
fig, ax = plt.subplots(figsize=(8, 4))
names, sizes = [], []
for dt in dtypes:
    t = torch.zeros(1, dtype=dt)
    names.append(str(dt).split('.')[-1])
    sizes.append(t.element_size() * 8)
ax.barh(names, sizes, color='teal', edgecolor='black')
ax.set_xlabel('bits per element'); ax.set_title('Tensor dtype → memory per element')
plt.tight_layout(); plt.show()

## CPU vs GPU

In [ ]:
x_cpu = torch.randn(3, 3)
x_gpu = x_cpu.to(device)
print(f"CPU: {x_cpu.device}  |  GPU: {x_gpu.device}")

# Timing demo (matrix multiply)
size = 2000
a = torch.randn(size, size)
b = torch.randn(size, size)
import time
t0 = time.perf_counter(); _ = a @ b; t_cpu = time.perf_counter() - t0
if device.type == 'cuda':
    ag, bg = a.to(device), b.to(device)
    torch.cuda.synchronize()
    t0 = time.perf_counter(); _ = ag @ bg; torch.cuda.synchronize(); t_gpu = time.perf_counter() - t0
    print(f"CPU matmul: {t_cpu:.3f}s | GPU matmul: {t_gpu:.3f}s")
else:
    print(f"CPU matmul: {t_cpu:.3f}s (no GPU in this session)")